In [0]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("MARVEL_RIVALS_API_KEY")
HEADERS = {"x-api-key": API_KEY}
BASE = "https://marvelrivalsapi.com/api"

In [0]:
# Sanity check: a known-simple endpoint (this was in your key's example)
r_maps = requests.get(f"{BASE}/v1/maps", headers=HEADERS)
print("Maps endpoint:", r_maps.status_code)
print(r_maps.text[:300])

In [0]:
# Step 1: Ask the API to go fetch / refresh this player's data
r0 = r0 = requests.get(f"{BASE}/v1/player/anonimis00/update", headers=HEADERS)
print("Update request:", r0.status_code, r0.text)

In [0]:
r1 = requests.get(f"{BASE}/v1/find-player/anonimis00", headers=HEADERS)
print("STATUS CODE: ", r1.status_code)
print("RAW DATA: ", repr(r1.text))
if r1.status_code == 200:
    print(json.dumps(r1.json(), indent=2))

In [0]:
# Get My Match history for alt account
UID = "676793468"

r2 = requests.get(f"{BASE}/v2/player/{UID}/match-history", headers=HEADERS)
print("STATUS CODE: ", r2.status_code)
print("RAW DATA: ", r2.text[:500])
if r2.status_code == 200:
    print(json.dumps(r2.json(), indent=2))

In [0]:
squad_usernames = ["Boorex", "Speeeeeeedwag", "LilRodneySon _", "Nobody_XVIII", "KennyKlock"]

for name in squad_usernames:
    r = requests.get(f"{BASE}/v1/find-player/{name}", headers=HEADERS)
    print(name, "->", r.status_code, r.text[:200])

In [0]:
MATCH_UID = "5513494_1780879146_2042395_11001_12"
r3 = requests.get(f"{BASE}/v1/match/{MATCH_UID}", headers=HEADERS)
print("STATUS CODE:", r3.status_code)
print("RAW DATA LENGTH:", len(r3.text))
print(r3.text[:1500])
if r3.status_code == 200:
    data = r3.json()
    print(list(data.keys()))

In [0]:
if r3.status_code == 200:
    data = r3.json()
    players = data['match_details']['match_players']
    print(f"Total players in match: {len(players)}")
    print("\nKeys available per player:")
    print(list(players[0].keys()))
    print("\nFirst player sample:")
    print(json.dumps(players[0], indent=2))

In [0]:
for player in players:
    hero_count = len(player['player_heroes'])
    if hero_count > 1:
        print(f"{player['nick_name']}: {hero_count} heroes played")
        print(json.dumps(player['player_heroes'], indent=2))

# DENG-001: Resolve Squad Usernames to UUIDs
**Goal**: call the `find-player` endpoint for each squad member (including yourself) and store the results in a dictionary mapping `username -> uid`. We'll use these UIDs in every subsequent API call, so this is the foundation everything else builds on.

## Inputs
- `BASE` (your API base URL)
- `HEADERS` (your API key header)
- `squad_usernames` (the list of 5 friends you already have)
- My username

## Expected output
A dictionary that looks like
```markdown
{
    "user_id_1": "uid_001",
    "user_id_2": "uid_002",
    "user_id_3": "uid_003"
}
```

## Requirements
1. Loop through all usernames - included mine - in one combined list
2. Only add to the dictionary if the status code is 200 (defensive check)
3. If a username comes back as anything other than 200, print a warning message so we know which one failed and why
4. Print the final dictionary at the end so we can verify

In [0]:
squad = ["anonimis00", "Boorex", "Speeeeeeedwag", "LilRodneySon _", "Nobody_XVIII", "KennyKlock"]
squad_dictionary = {}

for member in squad:
    squad_id_request = requests.get(f"{BASE}/v1/find-player/{member}", headers=HEADERS)
    if squad_id_request.status_code == 200:
        squad_dictionary[member] = squad_id_request.json()['uid']
    else:
        print("Unable to find squad member:", member, squad_id_request.status_code)

print(squad_dictionary)

# Ticket DENG-002: Pull Full Match History for Each Squad member
Using the UIDs from DENG-001, paginate through the match history endpoint for each player and collect every match across all season - saving the raw results into a local Python structure we will write in DENG-003.

## What is pagination?
APIs rarely return thousands of records in one shot - they return in "pages" (e.g. 10 or 20 records at a time) and give you a way to request the next page. Think of it like a book: you read page 1, then ask for page 2, and keep going until there are no more pages. For this API the match history endpoints supports a `page` query parameter starting at 1

## Inputs
- `squad_dictionary` from DENG-001

## Expected output: A dictionary structure like:
```markdown
{
    "anonimis00": [match1, match2, match3, ...],
    "Boorex": [match1, match2, ...],
    ...
}
```
Where each match is the raw dictionary object exactly as the API returned it - no modifications

### Requirements
1. For each player, start at `page=1` and keep requesting the next page until the API returns either a non-200 status code OR an empty match lis
2. Accumulate all pages into one flat lis per player - not a list of pages
3. Print progress as you go so you can see it working, e.g: `anonimis00 -> page -> 10 matches`
4. After all players are done, print a summary: each player's name and their total match count
5. Defensive check: If any page returns a non-200, print a warning and break out of that player's pagination loop (don't crash the whole script)

In [0]:
## Create print summary function
def squad_match_history_summary(match_histories):
    for squad_member, history in match_histories.items():
        print(f"{squad_member} -> {len(history)} matches")

In [0]:
import time
squad_match_history_data = {}

for member, uid in squad_dictionary.items():
    page = 1
    squad_match_history_data[member] = []
    while True:
        time.sleep(1)
        squad_match_history_response = requests.get(f"{BASE}/v2/player/{uid}/match-history?page={page}", headers=HEADERS)
        if squad_match_history_response.status_code != 200:
            print(f"Error: Received status code {squad_match_history_response.status_code} for squad member {member} at page {page}")
            break
        elif not squad_match_history_response.json()['match_history']:
            print(f"No match history for {member} at page {page}")
            break
        else:
            matches = squad_match_history_response.json()['match_history']
            squad_match_history_data[member].extend(matches)
            print(f"{member} -> page {page} -> {len(matches)} matches")
            print(f" running total for member {member}: {len(squad_match_history_data[member])}")
        page += 1

In [0]:
squad_match_history_summary(squad_match_history_data)

In [0]:
print(squad_match_history_data["Boorex"])

# Ticket DENG-003: Collect Unique Match UIDs for Detail Pulling
**Goal**: From `squad_match_history_data`, extract every unique `match_uid` across all players into a single deduplicated list. This is the list we'll use in the next ticket to pull full 12-player match details - and deduplication is critical because squad members who played together share the same `match_uid`.

**Why deduplication matters here**: If myself and Boorex played 30 matches together, without deduplication you'd call the match details API 60 times for those matches - wasting API calls and creating duplicate files in Bronze.

**Expected outcome:** A single Python list of unique match UIDs
Along with a print showing total matches across all players vs. unqiue match UIDs - the difference tells you roughly how many matches the squad played together.

**Requirements:**
1. Loop through every player's match history in squad_match_history_data
2. Extract the match_uid field from each match record
3. Deduplicate — each match_uid should appear exactly once
4. Print: Total match records: X | Unique match UIDs: Y

In [0]:
def total_records_and_uids_summary(squad_list, history_data, ids):
    total_records = 0
    for person in squad_list:
        total_records += len(history_data[person])
    print(f"Total match records: {total_records} | Unique match UIDs: {len(ids)}")

In [0]:
unique_id_set = set()

for squad_member, match_list in squad_match_history_data.items():
    for match in match_list:
        unique_id_set.add(match['match_uid'])

unique_match_ids = list(unique_id_set)

In [0]:
total_records_and_uids_summary(squad,squad_match_history_data,unique_match_ids)

# Ticket DENG-004: Pull Full Match Details for Every Unique Match UID
**Goal**: For each `match_uid` in `unique_match_ids`, call the match details endpoint and collect the full 12-player response. Store results in a dictionary `mapping match_uid → match details`. This gives us the team-level data we need for the performance rating model.

**Expected output:** A dictionary file:
```markdown
{
    "5513494_1780879146_...": { ...full match details... },
    "5513494_1780879201_...": { ...full match details... },
    ...
}
```

**Requirements:**
1. Loop through unique_match_ids and call the match details endpoint for each
2. Keep the time.sleep(1) pattern between every request — same rate limit applies
3. On 200, store the full .json() response in the dictionary keyed by match_uid
4. On non-200, print a warning with the match_uid and status code and continue to the next match (don't break — one failed match shouldn't stop the whole pull)
5. Print progress every 10 matches so you can see it working without flooding the console: Fetched 10/118 matches...
6. After the loop, print the final count: Successfully fetched X of 118 match details



In [0]:
match_details_per_id = {}
count = 0
for match_id in unique_match_ids:
    time.sleep(1)
    unique_match_response = requests.get(f"{BASE}/v1/match/{match_id}", headers=HEADERS)

    if unique_match_response.status_code != 200:
        print(f"Error: Received status code {unique_match_response.status_code} for match {match_id}")
        continue
    match_details_per_id[match_id] = unique_match_response.json()['match_details']
    count += 1
    if count % 10 == 0:
        print(f"Fetched {count}/{len(unique_match_ids)} matches...")
print(f"Successfully fetched {len(match_details_per_id)} of {len(unique_match_ids)} match details")

# Ticket DENG-005: Save Raw Data to Disk as JSON files
**Goal**: Write everything to local JSON files organized in the folder structure we designed earlier:

```markdown
marvel_rivals_data/
    match_history/
        anonimis00.json
        Boorex.json
        Speeeeeeedwag.json
        LilRodneySon _.json
        Nobody_XVIII.json
        KennyKlock.json
    match_details/
        5513494_1780879146_2042395_11001_12.json
        ... (one file per unique match_uid)
```
**Requirements**:

1. Create both folders if they don't already exist — there's a standard Python library for file/folder operations, do you know what it is?
2. Write each player's match history as a single JSON file named {username}.json inside match_history/
3. Write each match's details as a single JSON file named {match_uid}.json inside match_details/
4. Print confirmation as files are written: Saved match_history/anonimis00.json etc.
5. Print a final summary: Saved 6 player files and 118 match detail files

In [0]:
anon_map = {
    "anonimis00": "Player_You",
    "Boorex": "Player_A",
    "Speeeeeeedwag": "Player_B",
    "LilRodneySon _": "Player_C",
    "Nobody_XVIII": "Player_D",
    "KennyKlock": "Player_E",
}

In [0]:
with open('anonymized_map.json', "w") as json_file:
    json.dump(anon_map, json_file, indent=4)

In [0]:
import os
print(os.getcwd())

In [0]:
from pathlib import Path
Path("marvel_rivals_data/match_history").mkdir(parents=True, exist_ok=True)
Path("marvel_rivals_data/match_details").mkdir(exist_ok=True, parents=True)

In [0]:
base_file_path = Path("marvel_rivals_data/match_history")
for name, anon_name in anon_map.items():
    file_path = base_file_path / f"{anon_name}.json"
    with open(file_path, 'w') as json_file:
        json.dump(squad_match_history_data[name], json_file, indent=4)
    print(f"Saved match_history/{anon_name}.json")

In [0]:
details_file_path = Path("marvel_rivals_data/match_details")
for match_id, match_detail in match_details_per_id.items():
    new_file_path = details_file_path / f"{match_id}.json"
    with open(new_file_path, 'w') as json_file:
        json.dump(match_detail, json_file, indent=4)
    print(f"Saved match_details/{match_id}.json")

print(f"Saved {len(anon_map)} player files and {len(match_details_per_id)} match detail files")

In [0]:
import os

history_files = os.listdir("marvel_rivals_data/match_history")
detail_files = os.listdir("marvel_rivals_data/match_details")

print(f"match_history/ contains {len(history_files)} files: {history_files}")
print(f"match_details/ contains {len(detail_files)} files")
print(f"anonymized_map.json exists: {os.path.exists('anonymized_map.json')}")